In [ ]:
# 安装本 notebook 运行所需的额外依赖包(tiktoken、transformers、nbformat 等)
# pip install -r requirements-extra.txt

Comparing Various Byte Pair Encoding (BPE) Implementations


1. Using BPE from tiktoken

In [ ]:
# 从 importlib.metadata 导入 version 函数,用于查询已安装第三方库的版本号
from importlib.metadata import version
# 打印当前环境中 tiktoken 库的版本号,便于记录/复现实验环境
print("tiktoken version:", version("tiktoken"))

In [ ]:
# 导入 OpenAI 官方的 tiktoken 库:一个高效的 BPE 分词器实现(核心用 Rust 编写,速度快)
import tiktoken
# 加载与 GPT-2 模型配套的 BPE 编码器(内置词表 + 合并规则),得到可直接使用的分词器对象
tik_tokenizer=tiktoken.get_encoding('gpt2')
# 定义一段测试文本,后面会用不同的 BPE 分词器实现分别编码,用来对比它们的结果是否一致
text = "Hello, world. Is this-- a test?"

In [ ]:
# 用 tiktoken 分词器把文本编码为 token id 序列;
# allowed_special 显式允许 "<|endoftext|>" 这个特殊 token 字面量出现在输入中并被正常编码,
# 否则 tiktoken 默认会因检测到特殊 token 字符串而报错,以防止误用
integers = tik_tokenizer.encode(text, allowed_special={"<|endoftext|>"})
# 打印编码得到的 token id 列表
print(integers)

In [ ]:
# 将 token id 序列解码还原为原始文本,用于验证编码/解码是否可逆
strings = tik_tokenizer.decode(integers)

# 打印还原后的文本,理论上应与最初的 text 完全一致
print(strings)

In [ ]:
# 打印该分词器的词表大小(GPT-2 使用的 BPE 词表通常为 50257 个 token)
print(tik_tokenizer.n_vocab)

2. Using the original BPE implementation used in GPT-2

In [ ]:
# 导入 OpenAI 官方发布的原始 GPT-2 BPE 分词器实现(纯 Python 版本,非 tiktoken)
# get_encoder: 根据本地词表/合并规则文件构建编码器对象; download_vocab: 下载所需的词表文件
from bpe_openai_gpt2 import get_encoder, download_vocab

In [ ]:
# 下载 GPT-2 官方词表文件(encoder.json)与 BPE 合并规则文件(vocab.bpe)到本地目录
download_vocab()

In [ ]:
# 基于刚下载好的词表和合并规则文件,构建原始版 GPT-2 BPE 编码器对象
# model_name 指定存放词表文件的子目录名称, models_dir 指定该子目录所在的根路径
orig_tokenizer = get_encoder(model_name="gpt2_model", models_dir=".")

In [ ]:
# 用原始版 GPT-2 BPE 实现对同一段测试文本进行编码
integers = orig_tokenizer.encode(text)

# 打印结果,可与前面 tiktoken 的编码结果对比,验证两种实现是否一致
print(integers)

In [ ]:
# 将原始版 GPT-2 BPE 编码器输出的 token id 序列解码还原为文本
strings = orig_tokenizer.decode(integers)

# 打印还原后的文本,验证编码/解码的可逆性
print(strings)

3. Using the BPE via Hugging Face transformers

In [ ]:
# 导入 Hugging Face 的 transformers 库,用它内置的 GPT-2 分词器再做一次对比
import transformers

# 查看当前安装的 transformers 版本号
transformers.__version__

In [ ]:
# 导入 transformers 提供的“慢速版”(纯 Python 实现)GPT2Tokenizer
from transformers import GPT2Tokenizer

# 从 Hugging Face Hub 加载预训练的 GPT-2 分词器(会自动下载对应的词表与合并规则文件)
hf_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [ ]:
# 用 Hugging Face 慢速版分词器编码文本,并取出其中的 token id 列表(input_ids 字段)
hf_tokenizer(strings)["input_ids"]

In [ ]:
# 导入 transformers 提供的“快速版”分词器(底层基于 Rust 实现的 tokenizers 库,速度更快)
from transformers import GPT2TokenizerFast
# 加载预训练的快速版 GPT-2 分词器
hf_tokenizer_fast = GPT2TokenizerFast.from_pretrained("gpt2")

In [ ]:
# 用快速版分词器编码同一段文本,并取出 token id 列表,可与慢速版结果对比是否一致
hf_tokenizer_fast(strings)["input_ids"]

4. Using my own from-scratch BPE tokenizer

In [ ]:
# 导入操作系统路径处理、模块系统、内存文件流、Jupyter notebook 读取与动态创建模块所需的库
import os
import sys
import io
import nbformat
import types
# 定义一个工具函数:从另一个 notebook 文件中“导入”指定的函数/类定义,
# 这样就不用把 05_bpe-from-scratch 里手写的 BPETokenizerSimple 代码复制粘贴到这里
def import_from_notebook():
    # 内部辅助函数:根据 notebook 文件名(不含扩展名)和需要导入的函数/类名列表,
    # 读取该 notebook 并动态构建一个包含这些定义的模块对象
    def import_definitions_from_notebook(fullname, names):
        # 获取当前工作目录,用于拼接目标 notebook 的相对路径
        current_dir=os.getcwd()
        # 拼接出目标 notebook(bpe-from-scratch.ipynb)所在的完整路径
        path=os.path.join(current_dir,"..","05_bpe-from-scratch",fullname+".ipynb")
        # 规范化路径(处理掉路径中的 ".." 等部分)
        path=os.path.normpath(path)
         # Load the notebook
        # 如果目标 notebook 文件不存在,则直接抛出异常提示
        if not os.path.exists(path):
            raise FileNotFoundError(f"Notebook file not found at: {path}")
        # 以 UTF-8 编码打开该 notebook 文件
        with io.open(path,"r",encoding="utf-8") as f:
            # 用 nbformat 将 notebook 文件解析为可编程访问的对象(as_version=4 表示按 v4 格式解析)
            nb = nbformat.read(f, as_version=4)
        # Create a module to store the imported functions and classes
        # 动态创建一个新的模块对象,名字为 fullname,用来承载从 notebook 中提取出的函数/类
        mod=types.ModuleType(fullname)
        # 把这个动态模块注册到 sys.modules 中,这样后续 exec 执行的代码里如果有 import 也能找到它
        sys.modules[fullname]=mod
        # Go through the notebook cells and only execute function or class definitions
        # 遍历 notebook 中的每一个 cell
        for cell in nb.cells:
            # 只处理代码类型的 cell,跳过 markdown 等其他类型
            if cell.cell_type=='code':
                # 取出该 cell 的源代码文本
                cell_code=cell.source
                # 对每一个我们关心的目标名称(如 BPETokenizerSimple)逐一检查
                for name in names:
                     # Check for function or class definitions
                    # 简单地用字符串匹配判断该 cell 是否包含目标函数/类的定义
                    if f"def {name}" in cell_code or f"class {name}" in cell_code:
                        # 如果匹配到,就在动态模块的命名空间里执行这段代码,
                        # 从而把该函数/类定义“注入”到 mod 模块中,而不会执行 notebook 中的其他代码(如下载、画图等)
                        exec(cell_code, mod.__dict__)
        # 返回这个装载了目标函数/类定义的动态模块
        return mod
    # 指定要读取的目标 notebook 文件名(对应 05_bpe-from-scratch/bpe-from-scratch.ipynb)
    fullname = "bpe-from-scratch"
    # 指定需要从该 notebook 中提取出来的类名列表,这里只需要 BPETokenizerSimple 这一个类
    names = ["BPETokenizerSimple"]

    # 调用内部辅助函数完成导入,并返回构建好的模块对象
    return import_definitions_from_notebook(fullname, names)

In [ ]:
# 调用上面定义的 import_from_notebook(),从 05_bpe-from-scratch notebook 中动态加载出所需的类定义
imported_module=import_from_notebook()
# 从导入的模块中取出 BPETokenizerSimple 类(如果不存在则返回 None)
BPETokenizerSimple=getattr(imported_module,"BPETokenizerSimple", None)
# 实例化一个自己从零实现的 BPE 分词器对象
tokenizer_gpt2=BPETokenizerSimple()
# 加载 OpenAI 官方发布的 GPT-2 词表(encoder.json)和 BPE 合并规则(vocab.bpe),
# 让这个自制分词器复用与 GPT-2 完全相同的词表和合并规则,从而可以与前面几种实现做对比
tokenizer_gpt2.load_vocab_and_merges_from_openai(
     vocab_path=os.path.join("gpt2_model", "encoder.json"),
    bpe_merges_path=os.path.join("gpt2_model", "vocab.bpe")
)

In [ ]:
# 用自己从零实现的 BPE 分词器对同一段测试文本进行编码
integers = tokenizer_gpt2.encode(text)

# 打印结果,可与前面 tiktoken / 原始 OpenAI 实现 / Hugging Face 实现的编码结果做最终对比
print(integers)

5. A quick performance benchmark

In [ ]:
# 读取第 2 章用到的示例长文本《The Verdict》全文,作为性能基准测试(benchmark)的输入语料
with open("../01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

5.1 Original OpenAI GPT-2 tokenizer

In [ ]:
# 用 IPython 的 %timeit 魔法命令,测量“原始版 OpenAI GPT-2 BPE 实现”对整篇长文本编码的耗时
%timeit orig_tokenizer.encode(raw_text)

5.2 Tiktoken OpenAI GPT-2 tokenizer

In [ ]:
# 测量 tiktoken(Rust 实现)对同一篇长文本编码的耗时,预期会比纯 Python 实现快很多
%timeit tik_tokenizer.encode(raw_text)

5.3 Hugging Face OpenAI GPT-2 tokenizer

In [ ]:
# 测量 Hugging Face 慢速版分词器对长文本编码的耗时(不做截断,处理完整长度)
%timeit hf_tokenizer(raw_text)["input_ids"]

In [ ]:
# 与上一格相同,但加上 max_length 截断,限制最长编码长度为 5145,避免超长警告/性能开销,
# 用于对比“限制长度”前后是否会显著影响耗时
%timeit hf_tokenizer(raw_text, max_length=5145, truncation=True)["input_ids"]

In [ ]:
# 测量 Hugging Face 快速版(Rust 实现)分词器对长文本编码的耗时
%timeit hf_tokenizer_fast(raw_text)["input_ids"]

In [ ]:
# 快速版分词器同样加上 max_length 截断限制后的耗时测量,便于横向对比
%timeit hf_tokenizer_fast(raw_text, max_length=5145, truncation=True)["input_ids"]

5.4 My own GPT-2 tokenizer (for educational purposes)

In [ ]:
# 测量自己从零实现的 BPE 分词器(纯 Python,教学用途)对长文本编码的耗时,
# 预期会是这几种实现里最慢的,因为它没有做底层优化,主要用于理解算法原理
%timeit tokenizer_gpt2.encode(raw_text)